# Семинар 1. Чтение данных из S3

Курсовой проект ML/DLOps · Inside Airbnb

В этом ноутбуке нужно заполнить **две функции**. Импорты, клиент S3 и примеры вызова уже готовы.

1. Сохраните `01_read_data.ipynb` в корне своего репозитория.
2. Подставьте bucket и key, которые выдаст преподаватель ; локальная копия создаётся после первого чтения.
3. Выполняйте ячейки сверху вниз. Сначала заполните TODO 1, затем TODO 2.

Прежде чем реализовывать функцию, удалите `NotImplementedError`.

## Окружение и запуск

Команды выполняются **в терминале**, из папки репозитория:

```bash
pip install -r requirements.txt
```



In [11]:
# Зависимости устанавливаются через requirements.txt

## Импорты и настройки

Подставьте bucket и key, которые выдаст преподаватель.
Настройте переменные окружения через .env.
Не забудьте добавить файл в .gitignore

In [12]:
import os
from pathlib import Path
import boto3
import pandas as pd
from dotenv import load_dotenv
from botocore.exceptions import BotoCoreError, ClientError

load_dotenv()
S3_BUCKET = os.getenv("S3_BUCKET", "mlops26-course-bucket")
S3_KEY = os.getenv("S3_KEY", "prepared/listings.csv")
LOCAL_PATH = Path("data/listings.csv")
s3 = boto3.client("s3", endpoint_url=os.getenv("S3_ENDPOINT_URL", "https://storage.yandexcloud.net"), region_name=os.getenv("S3_REGION", "ru-central1"))

/Users/polina.sokol/Projects/airbnb-mlops/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


## TODO 1. Чтение CSV из S3

Заполните `read_s3_csv(bucket, key)`. Функция принимает имя bucket и полный ключ объекта и возвращает `DataFrame`.

Например, для `s3://course-bucket/prepared/listings.csv`:
- bucket: `course-bucket`;
- key: `prepared/listings.csv`.

После изменения определения функции выполните эту ячейку заново.

Ответ от s3 будет в `response["Body"]`.

In [13]:
#s3.get_object(
#        Bucket=<bucket>, Key=<key>
#    )

In [14]:
def read_s3_csv(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    with response["Body"] as body:
        return pd.read_csv(body, dtype={"listing_id": str})

### Первое чтение

Выполняйте после TODO 1. Проверьте первые строки, размер и названия колонок.
Если возникла ошибка S3, проверьте адрес и доступ к объекту.

In [15]:
df_s3 = read_s3_csv(S3_BUCKET, S3_KEY)
display(df_s3.head())
print("Размер таблицы:", df_s3.shape)
print("Колонки:", df_s3.columns.tolist())
LOCAL_PATH.parent.mkdir(exist_ok=True)
df_s3.to_csv(LOCAL_PATH, index=False)

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


Размер таблицы: (20151, 13)
Колонки: ['listing_id', 'snapshot_date', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'neighbourhood_cleansed', 'property_type', 'room_type', 'price']


## TODO 2. Обработка ошибок и локальная копия

Заполните `read_data`. При доступном S3 возвращаем удалённые данные.
При сбое доступа или сети читаем локальный CSV, если путь передан.
Сообщение о переходе к файлу должно быть видно в выводе ячейки.

Ошибки формата CSV должны оставаться видимыми.

In [16]:
def read_data(bucket, key, local_path=None):
    try:
        return read_s3_csv(bucket, key)
    except (BotoCoreError, ClientError, OSError) as error:
        if local_path is None:
            raise
        print(f"{type(error).__name__}: переход к локальному файлу {local_path}")
        return pd.read_csv(local_path, dtype={"listing_id": str})

### Проверка: S3 доступен

Ожидаем таблицу из S3 без сообщения о переходе к локальному файлу.

In [17]:
df = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH)
display(df.head())
print("Размер:", df.shape)

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


Размер: (20151, 13)


### Проверка: объект не найден

Намеренно меняем ключ. Ожидаем сообщение о переходе к `data/listings.csv` и таблицу из локальной копии.
Перед запуском убедитесь, что локальный файл существует.

In [18]:
df_local = read_data(S3_BUCKET, S3_KEY + ".missing", LOCAL_PATH)
display(df_local.head())
print("Размер:", df_local.shape)

NoSuchKey: переход к локальному файлу data/listings.csv


,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


Размер: (20151, 13)


### Дополнительная ручная проверка

Раскомментируйте **один** вызов за раз:

- нет объекта и не передан запасной путь — исходная ошибка S3;
- нет объекта и локального файла — `FileNotFoundError`.

Эти вызовы должны завершаться ошибкой. После проверки закомментируйте их обратно.

In [20]:
# read_data(S3_BUCKET, S3_KEY + ".missing")
# read_data(S3_BUCKET, S3_KEY + ".missing", "data/absent.csv")

## Перед pull request

- Обе функции заполнены, обычное чтение и переход к файлу проверены.
- В README указаны pip-команды, источник данных и ваша обычная среда открытия ноутбука.
- В `.gitignore` добавлены `data/` и `.ipynb_checkpoints/`.
- В коммит входят `01_read_data.ipynb`, README и `.gitignore`. CSV и ключи доступа остаются локально.
- В `.gitignore` есть файл secretkeys.py

На следующих занятиях продолжаем этот же курсовой проект.

In [21]:
df.info()

print("\nКоличество пропусков:")
print(df.isna().sum())

print("\nСтатистика целевой переменной:")
print(df["price"].describe())

print("\nПримеры значений price:")
print(df["price"].head(10))

print("\nКоличество уникальных объявлений:")
print(df["listing_id"].nunique())

print("\nКоличество дубликатов строк:")
print(df.duplicated().sum())

print("\nДаты сбора данных:")
print(df["snapshot_date"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20151 entries, 0 to 20150
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   listing_id              20151 non-null  int64  
 1   snapshot_date           20151 non-null  object 
 2   latitude                20151 non-null  float64
 3   longitude               20151 non-null  float64
 4   accommodates            20151 non-null  int64  
 5   bathrooms               20120 non-null  float64
 6   bedrooms                18603 non-null  float64
 7   beds                    19469 non-null  float64
 8   minimum_nights          20111 non-null  float64
 9   neighbourhood_cleansed  20151 non-null  object 
 10  property_type           20151 non-null  object 
 11  room_type               20151 non-null  object 
 12  price                   20151 non-null  float64
dtypes: float64(7), int64(2), object(4)
memory usage: 2.0+ MB

Количество пропусков:
listing_id 